In [ ]:
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox
import sqlite3
from datetime import datetime

# ================= DATABASE =================
DB_NAME = "classroom_complaints.db"
conn = sqlite3.connect(DB_NAME)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    role TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS complaints (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    classroom TEXT NOT NULL,
    type TEXT NOT NULL,
    description TEXT NOT NULL,
    date TEXT NOT NULL,
    status TEXT DEFAULT 'Pending'
)
""")

cur.execute("""
INSERT OR IGNORE INTO users(name, username, password, role)
VALUES ('Administrator', 'admin', 'admin123', 'admin')
""")
conn.commit()

# ================= DESIGN =================
NAVY = "#17365D"
BLUE = "#2874A6"
GREEN = "#239B56"
ORANGE = "#D68910"
RED = "#C0392B"
BG = "#EEF3F8"
WHITE = "#FFFFFF"
TEXT = "#243447"
MUTED = "#64748B"
BORDER = "#D8E1EA"

root = tk.Tk()
root.title("Classroom Care - Cleanliness Complaint System")
root.geometry("1100x700")
root.minsize(950, 620)
root.configure(bg=BG)

def clear():
    for widget in root.winfo_children():
        widget.destroy()

def header(title, subtitle=""):
    top = tk.Frame(root, bg=NAVY, height=112)
    top.pack(fill="x")
    top.pack_propagate(False)
    tk.Label(top, text="🏫  CLASSROOM CARE", bg=NAVY, fg="#B9DDF5",
             font=("Arial", 13, "bold")).pack(anchor="w", padx=28, pady=(12, 2))
    tk.Label(top, text=title, bg=NAVY, fg=WHITE,
             font=("Arial", 23, "bold")).pack(anchor="w", padx=28)
    if subtitle:
        tk.Label(top, text=subtitle, bg=NAVY, fg="#DCEAF5",
                 font=("Arial", 10)).pack(anchor="w", padx=30, pady=(2, 5))

def footer():
    tk.Label(root, text="Classroom Care  •  Report it. Track it. Improve it.",
             bg=BG, fg=MUTED, font=("Arial", 9)).pack(side="bottom", pady=7)

def card(parent, padding=16):
    return tk.Frame(parent, bg=WHITE, padx=padding, pady=padding,
                    highlightbackground=BORDER, highlightthickness=1)

def button(parent, text, command, color=BLUE, width=20):
    return tk.Button(parent, text=text, command=command, width=width,
                     bg=color, fg=WHITE, activebackground=color,
                     activeforeground=WHITE, relief="flat", bd=0,
                     font=("Arial", 10, "bold"), cursor="hand2",
                     padx=8, pady=9)

def setup_table_style():
    style = ttk.Style()
    style.theme_use("clam")
    style.configure("Treeview", font=("Arial", 10), rowheight=29,
                    background=WHITE, fieldbackground=WHITE, foreground=TEXT)
    style.configure("Treeview.Heading", font=("Arial", 10, "bold"),
                    background="#DCE8F2", foreground=NAVY, padding=7)
    style.map("Treeview", background=[("selected", "#C9E3F7")],
              foreground=[("selected", NAVY)])

def stat_card(parent, label, number, color):
    box = card(parent, 12)
    tk.Label(box, text=label, bg=WHITE, fg=MUTED,
             font=("Arial", 9, "bold")).pack()
    tk.Label(box, text=str(number), bg=WHITE, fg=color,
             font=("Arial", 23, "bold")).pack(pady=(5, 0))
    return box

def get_count(where="", params=()):
    cur.execute("SELECT COUNT(*) FROM complaints " + where, params)
    return cur.fetchone()[0]

# ================= HOME PAGE =================
def home():
    clear()
    hero = tk.Frame(root, bg=NAVY, height=205)
    hero.pack(fill="x")
    hero.pack_propagate(False)
    tk.Label(hero, text="🏫 CLASSROOM CARE", bg=NAVY, fg=WHITE,
             font=("Arial", 28, "bold")).pack(pady=(38, 8))
    tk.Label(hero, text="Classroom Cleanliness Complaint Management System",
             bg=NAVY, fg="#D6EAF8", font=("Arial", 13)).pack()
    tk.Label(hero, text="A cleaner classroom starts with one report.",
             bg=NAVY, fg=WHITE, font=("Arial", 11, "italic")).pack(pady=12)

    panel = card(root, 25)
    panel.pack(pady=32, ipadx=35)
    tk.Label(panel, text="Welcome!", bg=WHITE, fg=NAVY,
             font=("Arial", 19, "bold")).pack(pady=(3, 8))
    tk.Label(panel, text="Report classroom cleanliness issues and track their progress.\n"
                         "Students can submit reports and administrators can update status.",
             bg=WHITE, fg=MUTED, font=("Arial", 11),
             justify="center").pack(pady=(0, 18))
    button(panel, "🔐  Login", login_page, BLUE, 30).pack(pady=5)
    button(panel, "📝  Student Registration", register_page, GREEN, 30).pack(pady=5)
    button(root, "Exit", close_app, RED, 15).pack()
    footer()

# ================= REGISTRATION =================
def register_page():
    clear()
    header("Student Registration", "Create an account to report cleanliness problems")
    panel = card(root, 25)
    panel.pack(pady=25, ipadx=30)
    fields = {}
    for label in ("Full Name", "Username", "Password"):
        tk.Label(panel, text=label, bg=WHITE, fg=TEXT,
                 font=("Arial", 10, "bold")).pack(anchor="w", pady=(8, 4))
        ent = tk.Entry(panel, width=38, font=("Arial", 11), relief="solid", bd=1)
        if label == "Password":
            ent.config(show="*")
        ent.pack(ipady=6)
        fields[label] = ent

    def create_account():
        name = fields["Full Name"].get().strip()
        username = fields["Username"].get().strip()
        password = fields["Password"].get().strip()
        if not name or not username or not password:
            messagebox.showwarning("Missing Information", "Please fill in every field.")
            return
        try:
            cur.execute("INSERT INTO users(name,username,password,role) VALUES(?,?,?,'student')",
                        (name, username, password))
            conn.commit()
            messagebox.showinfo("Success", "Account created. You can now log in.")
            login_page()
        except sqlite3.IntegrityError:
            messagebox.showerror("Username Exists", "Please choose a different username.")

    button(panel, "Create Account", create_account, GREEN, 23).pack(pady=(18, 7))
    button(panel, "Back to Home", home, MUTED, 23).pack()
    footer()

# ================= LOGIN =================
def login_page():
    clear()
    header("Login", "Sign in to access your dashboard")
    panel = card(root, 25)
    panel.pack(pady=30, ipadx=30)

    tk.Label(panel, text="Username", bg=WHITE, fg=TEXT,
             font=("Arial", 10, "bold")).pack(anchor="w", pady=(5, 4))
    username_entry = tk.Entry(panel, width=38, font=("Arial", 11), relief="solid", bd=1)
    username_entry.pack(ipady=6)

    tk.Label(panel, text="Password", bg=WHITE, fg=TEXT,
             font=("Arial", 10, "bold")).pack(anchor="w", pady=(14, 4))
    password_entry = tk.Entry(panel, width=38, font=("Arial", 11),
                              show="*", relief="solid", bd=1)
    password_entry.pack(ipady=6)

    role = tk.StringVar(value="student")
    role_row = tk.Frame(panel, bg=WHITE)
    role_row.pack(pady=15)
    tk.Label(role_row, text="Login as:", bg=WHITE, fg=TEXT,
             font=("Arial", 10, "bold")).pack(side="left", padx=(0, 8))
    tk.Radiobutton(role_row, text="Student", variable=role, value="student",
                   bg=WHITE, activebackground=WHITE).pack(side="left", padx=5)
    tk.Radiobutton(role_row, text="Admin", variable=role, value="admin",
                   bg=WHITE, activebackground=WHITE).pack(side="left", padx=5)

    def login():
        username = username_entry.get().strip()
        password = password_entry.get().strip()
        if not username or not password:
            messagebox.showwarning("Missing Information", "Enter username and password.")
            return
        cur.execute("SELECT name,username,role FROM users WHERE username=? AND password=? AND role=?",
                    (username, password, role.get()))
        user = cur.fetchone()
        if not user:
            messagebox.showerror("Login Failed", "Invalid username, password, or role.")
        elif user[2] == "admin":
            admin_dashboard()
        else:
            student_dashboard(user[1], user[0])

    button(panel, "Login", login, BLUE, 23).pack(pady=(2, 7))
    button(panel, "Back to Home", home, MUTED, 23).pack()
    username_entry.focus_set()
    footer()

# ================= STUDENT DASHBOARD =================
def student_dashboard(username, name=None):
    clear()
    header("Student Dashboard", "Submit a complaint or track your reports")
    name = name or username

    welcome = card(root, 18)
    welcome.pack(fill="x", padx=30, pady=(18, 10))
    tk.Label(welcome, text=f"Welcome, {name} 👋", bg=WHITE, fg=NAVY,
             font=("Arial", 18, "bold")).pack(anchor="w")
    tk.Label(welcome, text="Thank you for helping maintain a clean learning environment.",
             bg=WHITE, fg=MUTED, font=("Arial", 10)).pack(anchor="w", pady=(4, 0))

    row = tk.Frame(root, bg=BG)
    row.pack(fill="x", padx=25, pady=5)
    stat_card(row, "MY COMPLAINTS", get_count("WHERE username=?", (username,)), BLUE).pack(
        side="left", expand=True, fill="x", padx=7)
    stat_card(row, "PENDING", get_count("WHERE username=? AND status='Pending'", (username,)), ORANGE).pack(
        side="left", expand=True, fill="x", padx=7)
    stat_card(row, "RESOLVED", get_count("WHERE username=? AND status='Resolved'", (username,)), GREEN).pack(
        side="left", expand=True, fill="x", padx=7)

    actions = tk.Frame(root, bg=BG)
    actions.pack(pady=22)
    for symbol, title, desc, btn_text, color, action in [
        ("📝", "Report a Problem", "Tell the administrator about\nan issue in your classroom.",
         "Submit Complaint", BLUE, lambda: complaint_page(username, name)),
        ("📋", "Track Complaints", "View your submitted reports\nand their current status.",
         "My Complaints", GREEN, lambda: my_complaints(username, name))
    ]:
        panel = card(actions, 18)
        panel.pack(side="left", padx=10, ipadx=8)
        tk.Label(panel, text=symbol, bg=WHITE, font=("Arial", 25)).pack()
        tk.Label(panel, text=title, bg=WHITE, fg=NAVY,
                 font=("Arial", 13, "bold")).pack(pady=6)
        tk.Label(panel, text=desc, bg=WHITE, fg=MUTED,
                 font=("Arial", 10), justify="center").pack(pady=5)
        button(panel, btn_text, action, color, 21).pack(pady=(10, 2))

    button(root, "Logout", home, RED, 16).pack(pady=4)
    footer()

# ================= SUBMIT COMPLAINT =================
def complaint_page(username, name=None):
    clear()
    header("Submit a Complaint", "Describe the issue so it can be addressed")
    panel = card(root, 22)
    panel.pack(pady=16, ipadx=30)

    tk.Label(panel, text="Classroom", bg=WHITE, fg=TEXT,
             font=("Arial", 10, "bold")).pack(anchor="w", pady=(3, 4))
    room = ttk.Combobox(panel, values=[
        "Room 101", "Room 102", "Room 103", "Room 104",
        "Computer Lab", "Electronics Lab"
    ], state="readonly", width=42)
    room.pack(ipady=4)

    tk.Label(panel, text="Complaint Type", bg=WHITE, fg=TEXT,
             font=("Arial", 10, "bold")).pack(anchor="w", pady=(12, 4))
    issue = ttk.Combobox(panel, values=[
        "Dirty Floor", "Dustbin Problem", "Dirty Desk", "Bad Odour",
        "Spillage", "Washroom Cleanliness", "Other"
    ], state="readonly", width=42)
    issue.pack(ipady=4)

    tk.Label(panel, text="Description", bg=WHITE, fg=TEXT,
             font=("Arial", 10, "bold")).pack(anchor="w", pady=(12, 4))
    description = tk.Text(panel, width=50, height=5, font=("Arial", 10),
                          relief="solid", bd=1, wrap="word")
    description.pack()

    def submit():
        details = description.get("1.0", "end").strip()
        if not room.get() or not issue.get() or not details:
            messagebox.showwarning("Missing Information", "Complete all fields before submitting.")
            return
        cur.execute(
            "INSERT INTO complaints(username,classroom,type,description,date,status) VALUES(?,?,?,?,?,?)",
            (username, room.get(), issue.get(), details,
             datetime.now().strftime("%Y-%m-%d %H:%M"), "Pending")
        )
        conn.commit()
        messagebox.showinfo("Submitted", "Your complaint has been recorded.")
        student_dashboard(username, name)

    buttons = tk.Frame(panel, bg=WHITE)
    buttons.pack(pady=15)
    button(buttons, "Submit Complaint", submit, GREEN, 19).pack(side="left", padx=5)
    button(buttons, "Back", lambda: student_dashboard(username, name), MUTED, 12).pack(side="left", padx=5)
    footer()

# ================= MY COMPLAINTS =================
def my_complaints(username, name=None):
    clear()
    header("My Complaints", "Filter and review your submitted reports")
    setup_table_style()

    controls = tk.Frame(root, bg=BG)
    controls.pack(fill="x", padx=22, pady=(12, 4))
    tk.Label(controls, text="Status:", bg=BG, fg=TEXT,
             font=("Arial", 10, "bold")).pack(side="left", padx=(0, 7))
    status_filter = tk.StringVar(value="All")
    filter_box = ttk.Combobox(controls, textvariable=status_filter,
                              values=["All", "Pending", "In Progress", "Resolved"],
                              state="readonly", width=16)
    filter_box.pack(side="left")

    columns = ("ID", "Classroom", "Type", "Description", "Date", "Status")
    frame = tk.Frame(root, bg=BG)
    frame.pack(fill="both", expand=True, padx=22, pady=8)
    tree = ttk.Treeview(frame, columns=columns, show="headings")
    widths = [50, 120, 145, 320, 140, 110]
    for col, width in zip(columns, widths):
        tree.heading(col, text=col)
        tree.column(col, width=width, anchor="w")
    tree.tag_configure("Pending", foreground=ORANGE)
    tree.tag_configure("In Progress", foreground=BLUE)
    tree.tag_configure("Resolved", foreground=GREEN)
    scroll = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    tree.configure(yscrollcommand=scroll.set)
    tree.pack(side="left", fill="both", expand=True)
    scroll.pack(side="right", fill="y")

    def load(*_):
        tree.delete(*tree.get_children())
        query = "SELECT id,classroom,type,description,date,status FROM complaints WHERE username=?"
        params = [username]
        if status_filter.get() != "All":
            query += " AND status=?"
            params.append(status_filter.get())
        query += " ORDER BY id DESC"
        cur.execute(query, params)
        for row in cur.fetchall():
            tree.insert("", "end", values=row, tags=(row[5],))

    filter_box.bind("<<ComboboxSelected>>", load)
    load()
    button(root, "Back to Dashboard", lambda: student_dashboard(username, name), MUTED, 22).pack(pady=(0, 10))
    footer()

# ================= ADMIN DASHBOARD =================
def admin_dashboard():
    clear()
    header("Admin Dashboard", "Manage complaints and monitor cleanliness progress")

    cards = tk.Frame(root, bg=BG)
    cards.pack(fill="x", padx=20, pady=25)
    stat_card(cards, "TOTAL COMPLAINTS", get_count(), NAVY).pack(side="left", expand=True, fill="x", padx=6)
    stat_card(cards, "PENDING", get_count("WHERE status='Pending'"), ORANGE).pack(side="left", expand=True, fill="x", padx=6)
    stat_card(cards, "IN PROGRESS", get_count("WHERE status='In Progress'"), BLUE).pack(side="left", expand=True, fill="x", padx=6)
    stat_card(cards, "RESOLVED", get_count("WHERE status='Resolved'"), GREEN).pack(side="left", expand=True, fill="x", padx=6)

    panel = card(root, 20)
    panel.pack(fill="x", padx=60, pady=10)
    tk.Label(panel, text="Complaint Management", bg=WHITE, fg=NAVY,
             font=("Arial", 16, "bold")).pack(pady=(4, 8))
    tk.Label(panel, text="Search reports, update complaint status, and review statistics.",
             bg=WHITE, fg=MUTED, font=("Arial", 10)).pack(pady=(0, 12))
    button(panel, "📋  View & Manage Complaints", all_complaints, BLUE, 32).pack(pady=5)
    button(panel, "📊  View Statistics", statistics, ORANGE, 32).pack(pady=5)
    button(root, "Logout", home, RED, 16).pack(pady=12)
    footer()

# ================= ADMIN: SEARCH AND UPDATE =================
def all_complaints():
    clear()
    header("All Complaints", "Search reports, filter by status, and update progress")
    setup_table_style()

    toolbar = tk.Frame(root, bg=BG)
    toolbar.pack(fill="x", padx=15, pady=(10, 5))
    tk.Label(toolbar, text="Search:", bg=BG, fg=TEXT,
             font=("Arial", 10, "bold")).pack(side="left", padx=(0, 5))
    search = tk.StringVar()
    tk.Entry(toolbar, textvariable=search, width=24, font=("Arial", 10)).pack(
        side="left", ipady=4, padx=(0, 12))
    tk.Label(toolbar, text="Status:", bg=BG, fg=TEXT,
             font=("Arial", 10, "bold")).pack(side="left", padx=(0, 5))
    status_filter = tk.StringVar(value="All")
    filter_box = ttk.Combobox(toolbar, textvariable=status_filter,
                              values=["All", "Pending", "In Progress", "Resolved"],
                              state="readonly", width=15)
    filter_box.pack(side="left")

    columns = ("ID", "Student", "Classroom", "Type", "Description", "Date", "Status")
    frame = tk.Frame(root, bg=BG)
    frame.pack(fill="both", expand=True, padx=15, pady=6)
    tree = ttk.Treeview(frame, columns=columns, show="headings")
    widths = [45, 100, 100, 130, 300, 130, 110]
    for col, width in zip(columns, widths):
        tree.heading(col, text=col)
        tree.column(col, width=width, anchor="w")
    tree.tag_configure("Pending", foreground=ORANGE)
    tree.tag_configure("In Progress", foreground=BLUE)
    tree.tag_configure("Resolved", foreground=GREEN)
    yscroll = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    xscroll = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
    tree.configure(yscrollcommand=yscroll.set, xscrollcommand=xscroll.set)
    tree.grid(row=0, column=0, sticky="nsew")
    yscroll.grid(row=0, column=1, sticky="ns")
    xscroll.grid(row=1, column=0, sticky="ew")
    frame.grid_rowconfigure(0, weight=1)
    frame.grid_columnconfigure(0, weight=1)

    def load(*_):
        tree.delete(*tree.get_children())
        query = "SELECT id,username,classroom,type,description,date,status FROM complaints WHERE 1=1"
        params = []
        if status_filter.get() != "All":
            query += " AND status=?"
            params.append(status_filter.get())
        if search.get().strip():
            pattern = "%" + search.get().strip() + "%"
            query += " AND (username LIKE ? OR classroom LIKE ? OR type LIKE ? OR description LIKE ? OR CAST(id AS TEXT) LIKE ?)"
            params.extend([pattern] * 5)
        query += " ORDER BY id DESC"
        cur.execute(query, params)
        for row in cur.fetchall():
            tree.insert("", "end", values=row, tags=(row[6],))

    search.trace_add("write", load)
    filter_box.bind("<<ComboboxSelected>>", load)
    load()

    bottom = tk.Frame(root, bg=BG)
    bottom.pack(fill="x", padx=15, pady=(3, 8))
    tk.Label(bottom, text="Set status:", bg=BG, fg=TEXT,
             font=("Arial", 10, "bold")).pack(side="left", padx=(0, 6))
    new_status = ttk.Combobox(bottom, values=["Pending", "In Progress", "Resolved"],
                              state="readonly", width=15)
    new_status.pack(side="left", padx=(0, 8))

    def update_status():
        selected = tree.selection()
        if not selected:
            messagebox.showwarning("Select Complaint", "Select a complaint from the table first.")
            return
        if not new_status.get():
            messagebox.showwarning("Select Status", "Choose a status first.")
            return
        complaint_id = tree.item(selected[0], "values")[0]
        cur.execute("UPDATE complaints SET status=? WHERE id=?",
                    (new_status.get(), complaint_id))
        conn.commit()
        load()
        messagebox.showinfo("Updated", f"Complaint #{complaint_id} is now {new_status.get()}.")

    button(bottom, "Update Status", update_status, ORANGE, 15).pack(side="left", padx=4)
    button(bottom, "Back to Dashboard", admin_dashboard, MUTED, 21).pack(side="right")
    footer()

# ================= STATISTICS =================
def statistics():
    clear()
    header("Complaint Statistics", "Overview of reported cleanliness issues")

    total = get_count()
    pending = get_count("WHERE status='Pending'")
    progress = get_count("WHERE status='In Progress'")
    resolved = get_count("WHERE status='Resolved'")

    row = tk.Frame(root, bg=BG)
    row.pack(fill="x", padx=22, pady=25)
    stat_card(row, "TOTAL", total, NAVY).pack(side="left", expand=True, fill="x", padx=7)
    stat_card(row, "PENDING", pending, ORANGE).pack(side="left", expand=True, fill="x", padx=7)
    stat_card(row, "IN PROGRESS", progress, BLUE).pack(side="left", expand=True, fill="x", padx=7)
    stat_card(row, "RESOLVED", resolved, GREEN).pack(side="left", expand=True, fill="x", padx=7)

    panel = card(root, 20)
    panel.pack(fill="x", padx=55, pady=10)
    tk.Label(panel, text="Complaint Status Breakdown", bg=WHITE, fg=NAVY,
             font=("Arial", 15, "bold")).pack(anchor="w", pady=(0, 12))

    items = [("Pending", pending, ORANGE),
             ("In Progress", progress, BLUE),
             ("Resolved", resolved, GREEN)]
    max_count = max([item[1] for item in items] + [1])
    for label, number, color in items:
        line = tk.Frame(panel, bg=WHITE)
        line.pack(fill="x", pady=6)
        tk.Label(line, text=label, width=14, anchor="w", bg=WHITE,
                 fg=TEXT, font=("Arial", 10, "bold")).pack(side="left")
        bar = tk.Canvas(line, height=20, bg="#E8EEF4", highlightthickness=0)
        bar.pack(side="left", fill="x", expand=True, padx=8)
        bar.update_idletasks()
        bar.create_rectangle(0, 0, int(350 * number / max_count), 20,
                             fill=color, outline=color)
        tk.Label(line, text=str(number), width=5, bg=WHITE, fg=TEXT,
                 font=("Arial", 10, "bold")).pack(side="left")

    button(root, "Back to Admin Dashboard", admin_dashboard, MUTED, 24).pack(pady=15)
    footer()

def close_app():
    try:
        conn.close()
    finally:
        root.destroy()

root.protocol("WM_DELETE_WINDOW", close_app)
home()
root.mainloop()
path = Path("/mnt/data/Classroom_Care_Improved.py")
path.write_text(code, encoding="utf-8")
print("Created:", path)


FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\Classroom_Care_Improved.py'